# 17 · 多 Agent：交出去的是任务，留下的是责任

协调器要分开管理逻辑 Agent、正在执行的实例以及驻留资源。取消请求发出后立刻腾槽，会让旧 worker 尚未停止时新工作已启动；
迟到完成没有 generation，则可能释放新一轮的槽位。我们先运行 AgentTree 状态机，再用真实 SQLite 协作板观察写入、通知与回执的不同提交点。
这些是合成 worker 身份，没有调用真正子模型，也没有创建 Codex 聊天。

**运行范围**：本章直接使用最终包中的共享组件，逐格验证本专题机制；不会启动模型、访问真实收费 API 或使用个人凭据。每次从第一格顺序运行，运行文件进入独立的 `runs/` 目录。改动参数后，从相应状态初始化格重新运行。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter
from unittest import TestCase
expect = TestCase()  # 只借用异常断言；下面逐格调用实现，不运行总测试替代实验。


## 源码事实与教学选择

固定对照提交：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。以下链接说明 Codex 对应机制；Python 类型和实验状态机是教学选择，运行它们不等于运行 Codex。

- [S24 · admit_start; request_shutdown](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/agent/control/runtime.rs#L286-L314)：全runtime关闭先关admission fence、cancel token及mailbox；admit_start前后复查fence；等待members耗尽后才定格失败报告。
- [B05 · post_inner](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/message-board/src/local.rs#L222-L250)：幂等键限定caller，fingerprint含操作及payload；事务内再查重复，防并发重复写；通知recipient在事务内计算并去掉作者。
- [B06 · post_inner](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/message-board/src/local.rs#L309-L364)：先提交post，再best effort通知；append立即返回；通知失败只记warn，不回滚已存post。


## 实验 1 · 授权求交集，执行与驻留分开限额


子 Agent 只能收紧父授权。execution_capacity 限正在运行的 lease，residency_capacity 限驻留的逻辑 Agent；
因此创建成功不保证立即能运行。改变两个容量，观察哪个动作先触及上限。


In [2]:
coord = load_example('17', 'coordinator')
tree = coord.AgentTree(execution_capacity=1, residency_capacity=2)
reader = tree.spawn('/root', 'reader', {'read','network'})
writer = tree.spawn('/root', 'writer', {'read','write'})
assert tree.agents[reader].grants == frozenset({'read'})
first = tree.start(reader)
with expect.assertRaisesRegex(RuntimeError, 'ExecutionCapacity'):
    tree.start(writer)
with expect.assertRaisesRegex(RuntimeError, 'ResidencyCapacity'):
    tree.spawn('/root','third', {'read'})
assert len(tree.leases) == 1
print({'reader_grants': sorted(tree.agents[reader].grants), 'running': list(tree.leases)})


{'reader_grants': ['read'], 'running': [Lease(agent='/root/reader', generation=1)]}


## 实验 2 · 旧实例的完成不能释放新 lease


先完成第一代，再启动第二代，然后注入第一代的重复完成。generation 是实例身份的一部分，
所以旧完成只入审计事件，不能把新实例标记为空闲。消息接受与启动也不同：运行时 followup 只入邮箱。


In [3]:
assert tree.finish(first, 'completed')
second = tree.start(reader)
assert second.generation == first.generation + 1
assert not tree.finish(first, 'late duplicate')
assert second in tree.leases and tree.agents[reader].state == 'running'
assert tree.followup(reader, 'Verify the changed file') is None
assert tree.agents[reader].mailbox[-1] == 'Verify the changed file'
assert tree.finish(second, 'completed')
# 容量被另一 Agent 占用时，followup 接受消息，启动单独失败。
occupied = tree.start(writer)
with expect.assertRaisesRegex(RuntimeError, 'ExecutionCapacity'):
    tree.followup(reader, 'Accepted input survives rejected admission')
assert 'Accepted input survives rejected admission' in tree.agents[reader].mailbox
assert tree.finish(occupied, 'completed')
print({'generations': [first.generation,second.generation], 'mailbox': tree.agents[reader].mailbox})


{'generations': [1, 2], 'mailbox': ['Verify the changed file', 'Accepted input survives rejected admission']}


## 实验 3 · 取消先封闭准入，停止确认后才释放容量


取消子树是一项收尾协议。这里 cancel_subtree 只做 fencing 并返回待停止的 leases，
宿主确认后才 finish；本状态机没有实际进程可杀，因此不能把 cancelling 当作进程已退出。


In [4]:
active = tree.start(reader)
pending = tree.cancel_subtree(reader)
assert pending == [active] and active in tree.leases
assert tree.agents[reader].state == 'cancelling'
with expect.assertRaisesRegex(RuntimeError, 'subtree_closed'):
    tree.send(reader, 'Too late')
with expect.assertRaisesRegex(RuntimeError, 'ExecutionCapacity'):
    tree.start(writer)
assert tree.finish(active, 'cancelled and cleanup confirmed')
assert not tree.agents[reader].resident
replacement = tree.start(writer)
assert tree.finish(replacement, 'completed')
print({'cancelled_state': tree.agents[reader].state, 'active_leases': len(tree.leases)})


{'cancelled_state': 'closed', 'active_leases': 0}


## 实验 4 · 协作板写入成功，通知仍可能失败


持久事实先提交，通知只是 best effort。让通知回调真实抛错，再模拟提交后丢回执：
调用异常不等于数据库没写，重试相同 `(actor, request_id)` 应返回原 seq，且不重播通知。


In [5]:
run_dir = fresh_run_dir('ch17')
notifications = []
def notify(actor, seq):
    notifications.append((actor,seq))
    raise ConnectionError('Synthetic recipient disconnected')
board = coord.Board(run_dir/'board.sqlite', notify)
board.subscribe('reviewer')
with expect.assertRaisesRegex(ConnectionError, 'response_lost_after_commit'):
    board.write('implementer','op-1','Patch ready: verify tests',writable=True,lose_response=True)
assert board.capture() == 1 and len(board.failures) == 1
seq = board.write('implementer','op-1','Patch ready: verify tests',writable=True)
assert seq == 1 and len(notifications) == 1
with expect.assertRaisesRegex(ValueError, 'request_id_reused'):
    board.write('implementer','op-1','Different patch',writable=True)
with expect.assertRaisesRegex(PermissionError, 'read_only'):
    board.write('reviewer','deny','Cannot write',writable=False)
print({'persisted': board.read(), 'notification_failures': board.failures})


{'persisted': [(1, 'implementer', 'op-1', 'Patch ready: verify tests', 'post')], 'notification_failures': [('reviewer', 1, 'ConnectionError')]}


## 实验 5 · 重开后按固定 cutoff 补读


补读让通知丢失不变成事实丢失。先捕获截止 seq，再新增记录，旧读取视图不应突然多出新内容；
下一次捕获才纳入新记录。这里提供记录补读，不是可靠消息队列、分布式共识或完全不可变的讨论快照。


In [6]:
cutoff = board.capture()
board.write('reviewer','review-1','Verified local assertion',writable=True,operation='append')
assert len(notifications) == 1  # append 不发送通知。
board.close()
board = coord.Board(run_dir/'board.sqlite', lambda actor,seq: None)
old_view = board.read(cutoff=cutoff)
new_view = board.read(after=cutoff)
assert len(old_view) == 1 and len(new_view) == 1
assert old_view[0][0] < new_view[0][0]
assert board.write('implementer','op-1','Patch ready: verify tests',writable=True) == 1
board.close()
print({'old_cutoff': cutoff, 'new_record_seq': new_view[0][0], 'durable_idempotency': True})


{'old_cutoff': 1, 'new_record_seq': 2, 'durable_idempotency': True}


## 你现在可以解释什么

能指出派工、消息送达、实际执行、完成确认、持久写入和通知各自的 owner。真实子模型、跨进程 lease 恢复和完整验证链仍未接入。

**验证边界**：本章断言验证当前局部实现与合成输入。未运行固定提交的 Rust 测试，未验证真实模型、外部服务或累计 Agent 的端到端行为。
